<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자연어 처리와 LLM · 09. RAG

## Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

- **원 논문:** [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- **저자 / 발표:** Patrick Lewis et al. · NeurIPS 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** retriever distribution과 RAG-Sequence marginalization을 재현한다.

**축소하거나 생략한 부분:** DPR+BART+Wikipedia 21M passages 대신 TF-IDF와 로컬 5문서를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1 Eq. (1): pη(z|x) | Task 1 softmax similarity | probability sum |
| §2.2 Eq. (2): RAG-Sequence | Task 2 marginalization | finite NLL |
| §2.2 Eq. (3): RAG-Token | Task 2 설명/비교 | per-token distinction |
| Fig. 1: retriever+generator | Task 3 | recall@1/cited answer |

## 핵심 아이디어와 수식

$$p_{RAG\text{-}Seq}(y\mid x)=\sum_{z\in top\text{-}k}p_{\eta}(z\mid x)p_{\theta}(y\mid x,z)$$

**기호 해설:** z는 문서, p_eta는 retriever, p_theta는 document-conditioned generator likelihood다.

**코드 대응:** Task 1이 TF-IDF retriever, Task 2가 RAGSequence class, Task 3이 NLL/recall을 구현한다.

**입력과 출력 shape:** query vectors [3,V], document vectors [5,V] → p(z|x) [3,5] → answer prob [3].

**포트폴리오 구현 범위:** neural DPR/BART 대신 결정적 surrogate를 쓰지만 latent-document marginalization은 정확하다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,json.loads,Path.read_text,torch.tensor,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

<details>
<summary><code>path.read_text(encoding=None, errors=None)</code></summary>

#### `path.read_text(encoding=None, errors=None)`

- **역할:** 텍스트 파일 전체를 문자열로 읽습니다.
- **입력·반환:** 인코딩 설정을 받고 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** 작은 설정·말뭉치·메타데이터 파일을 명시적 인코딩으로 읽습니다.
- **주의점:** 학습용 대용량 파일에는 전체 메모리 적재가 부담될 수 있습니다.
- **공식 문서:** [Path.read_text](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1 Eq. (1): pη(z|x)
- **노트북에서 구현할 부분:** Task 1 softmax similarity
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** probability sum

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,torch.zeros,Tensor.softmax,torch.stack,torch.log,F.normalize,torch.allclose,Tensor.sum,torch.ones -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 6개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>tensor.softmax(dim, dtype=None)</code></summary>

#### `tensor.softmax(dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** method 문법으로 attention score나 class logit을 정규화합니다.
- **주의점:** `dim`이 확률 합을 낼 축이며 입력은 일반적으로 raw logit이어야 합니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [Tensor.softmax](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html)

</details>

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>torch.log(input)</code></summary>

#### `torch.log(input)`

- **역할:** 각 원소에 자연로그를 적용합니다.
- **입력·반환:** 양수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 곱을 합으로 바꾸고 log-likelihood를 계산합니다.
- **주의점:** 0은 음의 무한대, 음수는 NaN이므로 작은 epsilon이나 clamp가 필요합니다.

##### 관련 수식과 코드 연결

$$
y_i=\log x_i,\qquad x_i>0
$$

- **기호:** $\log$는 자연로그입니다.
- **수식 → 코드:** API가 곱셈 관계를 덧셈 관계로 바꾸고 작은 확률의 크기를 안정적으로 표현합니다.
- **직관:** 확률 곱을 log-probability 합으로 계산할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같으며 입력은 양수여야 합니다.
- **공식 문서:** [torch.log](https://docs.pytorch.org/docs/stable/generated/torch.log.html)

</details>

<details>
<summary><code>F.normalize(input, p=2.0, dim=1, eps=1e-12)</code></summary>

#### `F.normalize(input, p=2.0, dim=1, eps=1e-12)`

- **역할:** 지정 축의 벡터 norm을 1로 정규화합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\hat x=\frac{x}{\max(\lVert x\rVert_p,\epsilon)}
$$

- **기호:** $p$는 norm 차수, $\epsilon$은 0으로 나누지 않기 위한 하한입니다.
- **수식 → 코드:** 코드의 `dim` 축에서 norm을 구한 뒤 모든 원소를 같은 값으로 나눕니다.
- **직관:** 벡터 크기를 제거해 방향이나 상대 패턴에 집중하게 합니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택 축의 norm은 약 1입니다.
- **공식 문서:** [F.normalize](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 1: 설정, tokenizer/count vector, TF-IDF retriever 분포를 구현하세요.
@dataclass
class RAGConfig:
    pass


def terms(text):
    raise NotImplementedError


def count_vector(text, vocabulary, term_to_id):
    raise NotImplementedError


def retrieval_distribution(query_vectors, document_vectors):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2 Eq. (2): RAG-Sequence
- **노트북에서 구현할 부분:** Task 2 marginalization
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** finite NLL

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 2: retriever와 generator likelihood를 marginalize하는 RAGSequence를 구현하세요.
class RAGSequence:
    def __init__(self, documents, config):
        raise NotImplementedError

    def generator_likelihood(self, queries, document_ids):
        raise NotImplementedError

    def forward(self, retrieval_probability, queries):
        raise NotImplementedError

    def grounded_answer(self, document_index):
        """Document id가 포함된 grounded answer를 반환한다."""
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Fig. 1: retriever+generator
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** recall@1/cited answer

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 3: marginal NLL, recall@1, grounded answer 평가를 구현하세요.
def rag_sequence_loss(sequence_probability):
    raise NotImplementedError


def evaluate_retrieval(similarity, documents, queries):
    raise NotImplementedError


def build_grounded_answers(model, top_indices):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.